# membox — Detailed Walkthrough (Simple → Advanced)

A progressive tour of the whole library. Each section builds on the previous one.

| # | Section | Concept |
|---|---------|---------|
| 1 | Episodic basics | `record`, `recent`, `search` |
| 2 | Importance & emotion | manual + auto scoring |
| 3 | Semantic facts | `learn`, reinforce, contradict |
| 4 | Temporal facts | `valid_from`/`valid_until`, recurrence |
| 5 | Retrieval scoring | R × V × I, `min_score`, decay |
| 6 | Context builder | sections, token budget |
| 7 | Procedural memory | triggers → actions |
| 8 | Conversation threads | `thread_id`, `summarize_thread` |
| 9 | Consolidation | episodes → facts |
| 10 | Reflection | higher-order patterns |
| 11 | Forgetting | tiered decay |
| 12 | `maintain()` | one-call housekeeping |
| 13 | Configuration & presets | `fast()` / `deep()` / custom |
| 14 | Editing & annotations | correct the record |
| 15 | Multi-user | `owner_id` isolation |
| 16 | Embeddings (optional) | semantic retrieval |
| 17 | LLM integration | importance scorer + chat loop |

> Uses in-memory DBs so nothing is written to disk. Swap `":memory:"` for a path to persist.


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from datetime import datetime, timedelta
from membox import Membox, MemoryConfig


## 1. Episodic memory — *what happened?*

Episodes are timestamped events with an importance score and optional emotion/source/metadata. This is the raw log of your agent's life.


In [2]:
m = Membox(":memory:")

m.record("User said they love hiking in the Himalayas", importance=0.8, source="conversation")
m.record("User ordered black coffee", importance=0.3, emotion="calm", source="order")
m.record("Rocky the dog had a vet checkup — all clear", importance=0.5, context={"pet": "Rocky"})

# Three ways to browse episodes
print("recent:", [e.content for e in m.recent(3)])
print("search:", [e.content for e in m.search("coffee")])
print("fields:", m.recent(1)[0].importance, m.recent(1)[0].context)


recent: ['Rocky the dog had a vet checkup — all clear', 'User ordered black coffee', 'User said they love hiking in the Himalayas']
search: ['User ordered black coffee']
fields: 0.5 {'pet': 'Rocky'}


## 2. Importance & emotion

Importance (0.0–1.0) drives two things: **retrieval ranking** and **forgetting** (critical memories never die). You can set it manually or let a scorer infer it.

| Mode | How |
|------|-----|
| Manual | pass `importance=` to `record()` |
| Rule-based auto | `MemoryConfig(auto_score_importance=True)` |
| LLM auto | inject an `LLMImportanceScorer` (see §17) |


In [3]:
from membox.importance import RuleBasedImportanceScorer

auto = Membox(":memory:", config=MemoryConfig(auto_score_importance=True))
ep = auto.record("I just got promoted to Director of Engineering!")
print(f"auto-scored  -> importance={ep.importance:.2f} emotion={ep.emotion}")

ep2 = auto.record("hey, what's up?")
print(f"small talk   -> importance={ep2.importance:.2f} emotion={ep2.emotion}")


auto-scored  -> importance=0.90 emotion=None
small talk   -> importance=0.30 emotion=None


## 3. Semantic facts — *what I know*

Facts are `(subject, predicate, object)` triples with a confidence score. `learn()` handles three cases automatically:

| Action | Trigger | Effect |
|--------|---------|--------|
| `new` | First time this triple is seen | Inserts with given confidence |
| `reinforced` | Same triple seen again | Confidence rises toward 1.0 |
| `contradicted` | Same subject+predicate, new object | Old fact deactivated, new one active |


In [4]:
s = Membox(":memory:")

print(s.learn("user", "name",    "Pranav",      confidence=0.95)[1])      # new
print(s.learn("user", "prefers", "black coffee", confidence=0.9)[1])     # new
print(s.learn("user", "prefers", "black coffee")[1])                     # reinforced
print(s.learn("user", "lives_in","Delhi", confidence=0.8)[1])            # new
print(s.learn("user", "lives_in","Mumbai")[1])                           # contradicted

print("\nActive facts about user:")
for f in s.about("user"):
    print(f"  {f.subject} {f.predicate} {f.object}  (conf={f.confidence:.2f}, active={f.is_active})")


new
new
reinforced
new
contradicted

Active facts about user:
  user name Pranav  (conf=0.95, active=True)
  user prefers black coffee  (conf=0.92, active=True)
  user lives_in Mumbai  (conf=0.50, active=True)


## 4. Temporal facts

Facts can have a **validity window** (`valid_from` / `valid_until`) and a **recurrence** pattern. `about(subject, at_time=...)` returns only facts that were true at that moment.


In [5]:
t = Membox(":memory:")

t.learn("user", "travels_to", "Tokyo",
        valid_from=datetime(2026, 7, 1), valid_until=datetime(2026, 7, 10),
        confidence=0.9)

before = datetime(2026, 6, 15)
during = datetime(2026, 7, 5)
after  = datetime(2026, 7, 20)

print("before trip:", [f.object for f in t.about("user", at_time=before)])
print("during trip:", [f.object for f in t.about("user", at_time=during)])
print("after  trip:", [f.object for f in t.about("user", at_time=after)])


before trip: []
during trip: ['Tokyo']
after  trip: []


## 5. Retrieval scoring

`recall()` combines three signals:

```
score = w_recency * R   +   w_relevance * V   +   w_importance * I
```

- **R (recency)** — `e^(-decay_rate * hours_ago)`, decays with time
- **V (relevance)** — keyword overlap with the query (or embedding sim, §16)
- **I (importance)** — the stored importance of the episode

Each `RetrievalResult` exposes the breakdown so you can debug rankings.


In [6]:
r = Membox(":memory:", config=MemoryConfig(w_recency=0.3, w_relevance=0.4, w_importance=0.3))

# Record at different importances
r.record("User loves hiking in the Himalayas", importance=0.9)
r.record("User mentioned a hiking trail once", importance=0.2)

for res in r.recall("hiking", k=3):
    print(f"score={res.score:.3f}  R={res.recency:.2f} V={res.relevance:.2f} I={res.importance:.2f}")
    print(f"   {res.episode.content}")

# min_score filters out weak noise
print("\nWith min_score=0.6:",
      [res.episode.content for res in r.recall("hiking", k=5, min_score=0.6)])


score=0.970  R=1.00 V=1.00 I=0.90
   User loves hiking in the Himalayas
score=0.760  R=1.00 V=1.00 I=0.20
   User mentioned a hiking trail once

With min_score=0.6: ['User loves hiking in the Himalayas', 'User mentioned a hiking trail once']


## 6. The context builder — your integration point

`context()` assembles a token-budgeted string with up to four sections:

1. **User Profile** — currently-valid facts
2. **Active Procedures** — matching if-then rules (§7)
3. **Relevant Memories** — top episodes for the query
4. **Patterns** — reflections (§10)

Paste the return value into your system prompt. That's the entire integration.


In [7]:
c = Membox(":memory:")
c.learn("user", "prefers", "black coffee", confidence=0.9)
c.learn("user", "name", "Pranav", confidence=0.95)
c.record("User got promoted to Director!", importance=1.0, emotion="ecstatic")
c.record("User ordered black coffee", importance=0.3)

print(c.context("what does the user like?", max_tokens=500))


## User Profile
- user name Pranav (95%)
- user prefers black coffee (90%)

## Relevant Memories
- (0m ago [ecstatic]) User got promoted to Director!
- (0m ago) User ordered black coffee


## 7. Procedural memory — *how I do things*

Procedures are if-then rules: *when trigger matches, do action*. They show up in the context's **Active Procedures** section so the LLM can act on them.


In [8]:
p = Membox(":memory:")
p.learn_procedure("goodnight", "dim the lights and lock the door", confidence=0.9)
p.learn_procedure("meeting starts", "mute notifications and open the agenda", confidence=0.8)

print("matched:", [(proc.trigger, proc.action) for proc in p.match_procedures("time to say goodnight")])
print("\nin context:")
print(p.context("goodnight"))


matched: [('goodnight', 'dim the lights and lock the door')]

in context:
## Active Procedures
- When 'goodnight' → dim the lights and lock the door (90%)


## 8. Conversation threads

Episodes can be grouped into a `thread_id` and nested via `parent_id`/`depth`. Long threads get compressed with `summarize_thread()` (pi-style compaction): older messages fold into one summary episode, recent ones stay verbatim.


In [9]:
th = Membox(":memory:")

# A short thread
th.record("Hi, I need help with a Python bug", thread_id="t1")
th.record("Sure, what's the error?", thread_id="t1")
th.record("It's a KeyError on 'user'", thread_id="t1")
print("thread episodes:", len(th.thread("t1")))

# Force a long thread so summarization kicks in
for i in range(6):
    th.record(" ".join([f"message-{i}"] * 60), thread_id="t2")

result = th.summarize_thread("t2")
print("summarized?", result.did_summarize)
print("summary preview:", (result.summary_episode.content[:80] + "...") if result.summary_episode else None)


thread episodes: 3
summarized? True
summary preview: ## Goal
Continue the thread spanning 2026-06-21 → 2026-06-21 (1 episodes summari...


## 9. Consolidation — episodes → facts

`consolidate()` scans unconsolidated episodes and extracts stable facts (e.g. *"I love coffee"* → `user prefers coffee`). The rule-based extractor matches common first-person patterns; swap in an LLM consolidator for production accuracy.

Episodes must be older than `consolidation_min_age_hours` (default 1h) to be eligible.


In [10]:
con = Membox(":memory:")
con.record("I love black coffee", importance=0.7)
con.record("I live in Mumbai", importance=0.6)
con.record("the weather is nice today", importance=0.2)  # no pattern → no fact

# Pretend time has passed so episodes are eligible
future = datetime.now() + timedelta(hours=2)
report = con.consolidate(now=future)
print("report:", {k: v for k, v in report.items() if k != "facts"})
print("facts learned:")
for f in con.about("user"):
    print(f"  {f.subject} {f.predicate} {f.object}  (conf={f.confidence:.2f})")


report: {'episodes_processed': 2, 'facts_extracted': 2}
facts learned:
  user prefers black coffee  (conf=0.70)
  user lives_in Mumbai  (conf=0.70)


## 10. Reflection — higher-order patterns

`reflect()` looks across many episodes and surfaces recurring patterns (e.g. *"user frequently mentions coffee"*). These feed the **Patterns** section of the context. Run it explicitly or enable `auto_reflect` in config.


In [11]:
ref = Membox(":memory:")
for _ in range(4):
    ref.record("User is stressed about the deadline again")

result = ref.reflect(episodes=ref.recent(10))   # explicit pass
print("evaluated:", result["evaluated"])
for r in result["reflections"]:
    print(f"  {r.subject} {r.predicate} {r.object}  (conf={r.confidence:.2f})")


evaluated: 4
  user frequently_mentions stressed  (conf=0.70)
  user frequently_mentions about  (conf=0.70)
  user frequently_mentions user  (conf=0.70)
  user frequently_mentions deadline  (conf=0.70)
  user frequently_mentions again  (conf=0.70)


## 11. Forgetting — tiered decay

`forget()` walks every episode and applies importance-tiered rules. Trivial memories fade in days; critical ones (importance ≥ 0.9) effectively never die. Actions are `delete`, `archive`, or `keep`.


In [12]:
fg = Membox(":memory:")

# A trivial old memory vs. a critical old memory
fg.record("User said 'lol'", importance=0.1, timestamp=datetime.now() - timedelta(days=30))
fg.record("User's mother passed away", importance=1.0, timestamp=datetime.now() - timedelta(days=30))

result = fg.forget(now=datetime.now())
print("summary:", {k: v for k, v in result.items() if k != "actions"})
for a in result["actions"]:
    print(f"  {a.action:8} imp={a.retention_score:.2f}  {a.reason}")


summary: {'deleted': 1, 'archived': 0, 'kept': 1, 'total_evaluated': 2}
  delete   imp=0.04  imp=0.1 ≤ 0.3, age=30d > 7d
  keep     imp=0.40  above all thresholds (imp=1.0, age=30d)


## 12. `maintain()` — one-call housekeeping

Runs consolidation → reflection → thread summarization → forgetting in dependency order. Call it periodically (e.g. after every N turns) so you don't have to orchestrate each step.


In [13]:
mt = Membox(":memory:")
for _ in range(3):
    mt.record("I love black coffee")
for i in range(6):
    mt.record(" ".join([f"chat-{i}"] * 60), thread_id="t1")

report = mt.maintain(now=datetime.now() + timedelta(hours=2))
print("keys:", list(report.keys()))
print("consolidate episodes_processed:", report["consolidate"]["episodes_processed"])
print("forget summary:", {k: v for k, v in report["forget"].items() if k != "actions"})


keys: ['consolidate', 'summarized_threads', 'forget']
consolidate episodes_processed: 3
forget summary: {'deleted': 0, 'archived': 0, 'kept': 9, 'total_evaluated': 9}


## 13. Configuration & presets

Every knob lives in `MemoryConfig`. Override what you need, or use a preset:

| Preset | Use case | Behavior |
|--------|----------|----------|
| `MemoryConfig()` (default) | General | balanced |
| `MemoryConfig.fast()` | Chatbots | aggressive forgetting, small context |
| `MemoryConfig.deep()` | Personal assistants | long retention, embeddings on |


In [14]:
fast = Membox(":memory:", config=MemoryConfig.fast())
deep = Membox(":memory:", config=MemoryConfig.deep())

# Custom: emphasise relevance, forget slowly
custom = MemoryConfig(decay_rate=0.005, w_relevance=0.6, w_recency=0.2, w_importance=0.2,
                      max_context_tokens=4000)
cust = Membox(":memory:", config=custom)
print("fast decay:", fast._config.decay_rate)
print("deep decay:", deep._config.decay_rate)
print("custom weights: R V I =", cust._config.w_recency, cust._config.w_relevance, cust._config.w_importance)


fast decay: 0.1
deep decay: 0.005
custom weights: R V I = 0.2 0.6 0.2


/var/folders/tz/j7rs6s3j4px7cdgmy_ptcgk40000gn/T/ipykernel_60028/3906746704.py:2: RuntimeWarning: sentence-transformers not installed. Embedding model 'all-MiniLM-L6-v2' disabled. Install with: pip install sentence-transformers
  deep = Membox(":memory:", config=MemoryConfig.deep())


## 14. Editing & annotations — correct the record

Mistakes happen. You can:
- `update_episode()` — edit an episode in place
- `annotate_episode()` — append a timestamped correction/note (audit trail, original intact)
- `edit_fact()` / `correct_fact()` — fix or supersede a fact


In [15]:
ed = Membox(":memory:")
ep = ed.record("User lives in Paris")

updated = ed.update_episode(ep.id, content="User lives in Lyon", importance=0.8)
print("after update:", ed.recent(1)[0].content)

annotated = ed.annotate_episode(ep.id, correction="actually it's Lyon, not Paris")
print("annotations:", annotated.context.get("__annotations__"))

# Facts
f, _ = ed.learn("user", "job", "engineer", confidence=0.7)
fixed, action = ed.correct_fact(f.id, new_object="staff engineer")
print(f"correct_fact action={action}: {fixed.subject} {fixed.predicate} {fixed.object}")


after update: User lives in Lyon
annotations: [{'timestamp': '2026-06-21T21:04:53.196487', 'correction': "actually it's Lyon, not Paris"}]
correct_fact action=corrected: user job staff engineer


## 15. Multi-user isolation

Pass `owner_id` to partition memory per user/agent. Each owner only sees its own episodes, facts, procedures, and reflections in the same database file.


In [16]:
import tempfile, os
db = os.path.join(tempfile.mkdtemp(), "multi.db")

alice = Membox(db, owner_id="alice")
bob   = Membox(db, owner_id="bob")

alice.record("Alice loves rock climbing", importance=0.8)
bob.record("Bob prefers baking", importance=0.6)

# Alice's recall sees only her memories
print("alice sees:", [r.episode.content for r in alice.recall("hobbies", k=5)])
print("bob   sees:", [r.episode.content for r in bob.recall("hobbies", k=5)])


alice sees: ['Alice loves rock climbing']
bob   sees: ['Bob prefers baking']


## 16. (Optional) Embeddings — true semantic retrieval

Keyword overlap misses synonyms ("hiking" vs "outdoor activities"). Enable an embedding model for semantic similarity:

```bash
uv sync --extra embeddings
```

Set `embedding_model_name` in config and `Membox` will persist embeddings in SQLite and use them in `recall()` / `context()`. Hybrid scoring blends embeddings with keyword overlap via `w_embedding` / `w_keyword`.


In [17]:
try:
    cfg = MemoryConfig(embedding_model_name="all-MiniLM-L6-v2")
    em = Membox(":memory:", config=cfg)
    em.record("I enjoy long walks in the mountains", importance=0.7)
    em.record("I love debugging Python code", importance=0.5)

    # Synonym query: keyword overlap is low, embeddings rescue it
    for r in em.recall("outdoor activities", k=2):
        print(f"score={r.score:.3f} | {r.episode.content}")
except ImportError as e:
    print("Embeddings extra not installed. Install with: uv sync --extra embeddings")
    print("(", str(e)[:80], "...)")


score=0.510 | I enjoy long walks in the mountains
score=0.450 | I love debugging Python code


/var/folders/tz/j7rs6s3j4px7cdgmy_ptcgk40000gn/T/ipykernel_60028/2860186838.py:3: RuntimeWarning: sentence-transformers not installed. Embedding model 'all-MiniLM-L6-v2' disabled. Install with: pip install sentence-transformers
  em = Membox(":memory:", config=cfg)


## 17. LLM integration — the full loop

Two extension points connect `membox` to a real LLM:

1. **`LLMImportanceScorer`** — infer importance + emotion on every `record()` (works with any OpenAI-compatible client).
2. **The chat loop** — `record` → `context` → `LLM` → `record`. Framework-agnostic.

Below is a self-contained example using the rule-based scorer (no API key needed). Swap in `LLMImportanceScorer(client, model=...)` and a real client to go live.


In [18]:
from membox.importance import RuleBasedImportanceScorer

# 1) Auto importance via the rule-based scorer (no API key)
scorer = RuleBasedImportanceScorer()
agent = Membox(":memory:", config=MemoryConfig(auto_score_importance=True))

# Pre-seed some preferences
agent.learn("user", "name", "Pranav", confidence=0.95)
agent.learn("user", "prefers", "black coffee", confidence=0.9)

# 2) The chat loop (mocked LLM; replace `my_llm` with a real client call)
def my_llm(system_prompt: str, user_message: str) -> str:
    # In production: client.chat.completions.create(model=..., messages=[...])
    return f"[mocked reply using context of {len(system_prompt)} chars]"

def chat(user_message: str) -> str:
    ep = agent.record(user_message)                       # auto-scored
    context = agent.context(user_message)
    system_prompt = f"You are a helpful assistant.\n\n{context}"
    reply = my_llm(system_prompt, user_message)
    agent.record(f"Assistant: {reply}", importance=0.2, source="response")
    return reply

print("turn 1:", chat("I just got promoted to Director!")[:60], "...")
print("turn 2:", chat("What do you know about me?")[:60], "...")
print("\nfinal memory:\n", agent.context("who is the user?"))


turn 1: [mocked reply using context of 170 chars] ...
turn 2: [mocked reply using context of 272 chars] ...

final memory:
 ## User Profile
- user name Pranav (95%)
- user prefers black coffee (90%)

## Relevant Memories
- (0m ago) I just got promoted to Director!
- (0m ago) What do you know about me?
- (0m ago) Assistant: [mocked reply using context of 272 chars]
- (0m ago) Assistant: [mocked reply using context of 170 chars]


## 📚 Cheatsheet

| Goal | Call |
|------|------|
| Store an event | `memory.record(content, importance=, emotion=, source=, thread_id=)` |
| Browse recent events | `memory.recent(n)`, `memory.search(keyword)` |
| Store a fact | `memory.learn(subject, predicate, obj, confidence=)` |
| Query facts | `memory.about(subject, at_time=)`, `memory.find_fact(subject, predicate=)` |
| Retrieve memories | `memory.recall(query, k=, min_score=)` |
| Build prompt context | `memory.context(query, max_tokens=)` |
| Store a routine | `memory.learn_procedure(trigger, action, confidence=)` |
| Compress episodes → facts | `memory.consolidate()` / `consolidate_all()` |
| Compress a thread | `memory.summarize_thread(thread_id)` |
| Surface patterns | `memory.reflect()` |
| Prune stale memories | `memory.forget()` |
| Run all maintenance | `memory.maintain()` |
| Edit an episode | `memory.update_episode(id, ...)`, `annotate_episode(id, ...)` |
| Fix a fact | `memory.edit_fact(id, ...)`, `correct_fact(id, ...)` |
| Health check | `memory.stats()` |
| Per-user isolation | `Membox(db, owner_id="alice")` |
| Semantic retrieval | `MemoryConfig(embedding_model_name="all-MiniLM-L6-v2")` |
| Auto importance | `MemoryConfig(auto_score_importance=True)` or `LLMImportanceScorer` |

**Further reading:** `README.md`, `demos/demo_openrouter.py`, `lessons/` (builds the library from scratch), `tests/` (executable spec).
